# Block (Vanilla) BSF — toma *easy*

Featurize a sparse mixture of superposed manifolds with a **block BSF (free encoder + per-sample block TopK)** and score per-concept recovery against the ground truth (toma knows the true `mᵢ` and which concepts were active).

A *concept* is a block of `group_size` latent dims; `group_size=3` covers the curved manifolds (ambient dim up to 3).

This uses a *less-superposed* ("easy") config — `d=128`, `M=128` concepts (half
of them curved multi-dim manifolds), `l0=4` active per sample. There the three
block-sparse featurizers land near the oracle ceiling, while a classical
single-atom SAE *shatters* the curved manifolds. See the sibling notebooks for
the other featurizers: `01_block`, `02_grassmannian`, `03_group_lasso`, `04_sae`.

> Trains on 300k samples — run on a GPU (~a few min); CPU works but is slow.

In [ ]:
%config InlineBackend.figure_format = 'retina'
import numpy as np
import torch
import matplotlib.pyplot as plt

import toma
from toma import bsf
from toma.configs import BenchmarkConfig, _manifold_pool
from toma.metrics import (best_group_per_component, group_oracle_r2,
                          component_f1, oracle_k_r2)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch:', torch.__version__)

## 1. Build the toma benchmark and sample data

In [ ]:
# A less-superposed ("easy") config: low l0=4, M=d=128 (only slightly
# overcomplete), and HALF the concepts curved multi-dim manifolds (sphere,
# torus, …). This is the regime where a block-sparse featurizer recovers the
# manifolds near the oracle ceiling while a classical single-atom SAE shatters
# them. (The canonical, harder SMALL/MEDIUM/… scales live in `toma.configs`.)
cfg = BenchmarkConfig(
    name='easy', d=128,
    manifolds=_manifold_pool(128, concept_frac=0.5),
    n_train=300_000, n_test=30_000, l0=4, noise=0.0,
    sae_width=512, sae_k=12,
)

dgp   = cfg.build_dgp()
train = dgp.sample(n=cfg.n_train, l0=cfg.l0, noise=cfg.noise)
test  = dgp.sample(n=cfg.n_test,  l0=cfg.l0, noise=cfg.noise)

# Scale so mean ||x||^2 = 1 (toma convention); scale the contributions the same.
scale = float(np.sqrt((train['x'] ** 2).sum(-1).mean()))
x_tr  = train['x'] / scale
x_te  = test['x']  / scale
m_te  = test['contributions'] / scale          # (N, M, d) true per-concept mᵢ
masks = test['active_masks']                    # (N, M) bool

from collections import Counter
print(f'cfg={cfg.name}  d={cfg.d}  M={cfg.n_components}  l0={cfg.l0}  scale={scale:.3f}')
print(f'x_train {x_tr.shape}   x_test {x_te.shape}')
print('manifold mix:', dict(Counter(c.mtype for c in dgp.components)))

## 2. Manifold zoo & mixture

In [ ]:
# Helper: 2-D PCA in numpy (no sklearn dep)
def _pca2(X):
    X = X - X.mean(0)
    _, _, Vt = np.linalg.svd(X, full_matrices=False)
    return X @ Vt[:2].T

COLORS = {
    'Circle': '#6366f1', 'Sphere': '#f43f5e', 'Torus': '#fbbf24',
    'Mobius': '#10b981', 'SwissRoll': '#8b5cf6', 'Helix': '#ec4899',
    'FlatDisk': '#06b6d4', 'Concept': '#fb923c',
}

try:
    ZCMAP = plt.get_cmap('managua')       # preferred; fallback to viridis
except ValueError:
    ZCMAP = plt.get_cmap('viridis')

# One representative per manifold type
seen, reps = {}, []
for comp in dgp.components:
    if comp.mtype not in seen:
        seen[comp.mtype] = comp
        reps.append(comp)

ncols = min(len(reps), 5)
nrows = -(-len(reps) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.2, nrows * 3.2), squeeze=False)
axes = axes.flatten()

for ax, comp in zip(axes, reps):
    mask  = masks[:, comp.idx]
    mi    = m_te[mask, comp.idx]
    theta = test['thetas'][comp.idx][:, 0]
    t_norm = (theta - theta.min()) / (theta.max() - theta.min() + 1e-9)
    xy = _pca2(mi)
    ax.scatter(xy[:, 0], xy[:, 1], c=t_norm, cmap=ZCMAP, s=3, alpha=0.6, rasterized=True)
    ax.set_title(f'{comp.mtype}  (k={comp.ambient_dim})', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_box_aspect(1)                             # square panel
    ax.set_aspect('equal', adjustable='datalim')     # true shapes: 1-D Concept -> a line

for ax in axes[len(reps):]:
    ax.set_visible(False)

plt.suptitle('Manifold zoo — PCA(mᵢ) per type, colored by intrinsic θ', fontsize=11)
plt.tight_layout()
plt.show()

# Mixture: PCA of x colored by dominant mᵢ type
norms_dom = np.linalg.norm(m_te, axis=2)
norms_dom[~masks] = 0
dominant   = norms_dom.argmax(1)
dom_colors = [COLORS.get(dgp.components[i].mtype, '#888') for i in dominant]

xy_mix = _pca2(x_te[:3000])
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(xy_mix[:, 0], xy_mix[:, 1], c=dom_colors[:3000], s=1.5, alpha=0.4, rasterized=True)
for mtype, c in COLORS.items():
    ax.scatter([], [], c=c, s=20, label=mtype)
ax.legend(loc='upper right', fontsize=7, frameon=False)
ax.set_title('Mixture x — PCA colored by dominant mᵢ type')
ax.set_xticks([]); ax.set_yticks([])
ax.set_box_aspect(1)
ax.set_aspect('equal', adjustable='datalim')
plt.tight_layout()
plt.show()

## 3. Train the block BSF (free encoder + per-sample block TopK)

In [ ]:
# group_size=3 covers the curved manifolds (ambient dim up to 3); l0=cfg.l0
# matches the DGP's true active count; the free encoder exploits an OVERCOMPLETE
# dictionary (n_groups=2M). Converges by ~40 epochs on this config.
model = bsf.VanillaBSF(d=cfg.d, n_groups=2 * cfg.n_components, group_size=3, l0=cfg.l0)
bsf.train(model, x_tr, epochs=60, lr=4e-4, snr=0.1, device=DEVICE)   # train as a denoiser (10% input noise)

## 4. Score recovery against the ground truth

In [ ]:
# Encode the test set -> (N, G, K) codes, flatten to (N, G*K) and grab the
# (G*K, d) decoder atoms so the toma metrics can score per-concept recovery.
with torch.no_grad():
    z = model.encode(torch.as_tensor(x_te, dtype=torch.float32, device=DEVICE))
codes = z.reshape(z.shape[0], -1).cpu().numpy()           # (N, G*K)
dictionary = model.decoder_atoms().detach().cpu().numpy()  # (G*K, d)

r2_recon = bsf.recon_r2(model, torch.as_tensor(x_te, dtype=torch.float32, device=DEVICE))

# Block-aware recovery: assign each concept to the GROUP whose firing best
# predicts it (group F1), then reconstruct the true mᵢ from only that group's
# atoms (group-oracle R²). This respects the block structure -- the atom-level
# oracle_k_r2 would mismatch here by picking the global top-k atoms.
best_g, f1 = best_group_per_component(codes, model.n_groups, model.group_size, masks, dgp.components)
r2 = group_oracle_r2(codes, dictionary, model.group_size, m_te, masks, dgp.components, best_groups=best_g)

print(f'recon R²                 {r2_recon:.4f}')
print(f'group-oracle R² (mean)   {np.nanmean(r2):.4f}')
print(f'group F1        (mean)   {np.nanmean(f1):.4f}')

## 5. Per-concept recovery

In [ ]:
# Per-concept recovery: R² and F1, sorted, with the mean marked. Concepts to the
# right are recovered cleanly; the left tail is where the featurizer struggles
# (heavy superposition / curved manifolds).
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for a, vals, title in [(ax[0], np.sort(r2[~np.isnan(r2)]), 'per-concept R²'),
                       (ax[1], np.sort(f1[~np.isnan(f1)]), 'per-concept F1')]:
    a.plot(vals, lw=2, color='#6366f1')
    a.axhline(vals.mean(), ls='--', color='#64748b', label=f'mean {vals.mean():.3f}')
    a.set_title(title); a.set_xlabel('concept (sorted)')
    a.set_ylim(min(0.0, float(vals.min())) - 0.05, 1.02)
    a.legend(loc='lower right'); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Recovery in 3D — the manifolds, true vs recovered

In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
try:
    CMAP = plt.get_cmap('managua')       # preferred diverging map (mpl >= 3.10)
except ValueError:
    CMAP = plt.get_cmap('viridis')

def style_panel(ax, p, col, cmap, s=7.0, core_alpha=0.6):
    'Light floor + soft contact shadow + translucent halo under a translucent core.'
    p = p - p.mean(0)
    Rmax = float(np.percentile(np.hypot(p[:, 0], p[:, 1]), 99) * 1.05) + 1e-9
    zlo, zhi = float(p[:, 2].min()), float(p[:, 2].max()); zspan = max(zhi - zlo, 1e-6)
    fr = 1.45 * Rmax
    floor_z = zlo - max(0.35 * zspan, 0.30 * Rmax)     # floored to in-plane size so
    z_top   = zhi + max(0.05 * zspan, 0.05 * Rmax)     # planar manifolds still get space
    ax.computed_zorder = False
    quad = [[(-fr, -fr, floor_z), (fr, -fr, floor_z), (fr, fr, floor_z), (-fr, fr, floor_z)]]
    ax.add_collection3d(Poly3DCollection(quad, facecolor='#eef1f4', edgecolor='none', alpha=0.9, zorder=0))
    ax.scatter(p[:, 0], p[:, 1], np.full(len(p), floor_z), c=[(0.08, 0.08, 0.12)],
               s=s * 2.6, alpha=0.03, linewidths=0, depthshade=False, zorder=1)   # shadow
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], c=col, cmap=cmap, s=s * 2.0,
               alpha=0.1, linewidths=0, depthshade=False, zorder=2)               # halo
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], c=col, cmap=cmap, s=s,
               alpha=core_alpha, linewidths=0, depthshade=False, zorder=3)        # core
    # TRUE equal aspect (no distortion): equal DATA limits on all 3 axes (a cube
    # spanning points + floor), then ax.set_aspect('equal').
    xs = np.r_[p[:, 0], -fr, fr]; ys = np.r_[p[:, 1], -fr, fr]; zs = np.r_[p[:, 2], floor_z, z_top]
    cx, cy, cz = xs.mean(), ys.mean(), zs.mean()
    R = max(np.ptp(xs), np.ptp(ys), np.ptp(zs)) / 2 + 1e-9
    ax.set_xlim(cx - R, cx + R); ax.set_ylim(cy - R, cy + R); ax.set_zlim(cz - R, cz + R)
    ax.set_axis_off()
    ax.set_aspect('equal')

# Matched unit -> per-concept reconstruction. Block featurizers: that concept's
# best group. SAE (group_size=1): its k_i most concept-discriminative atoms.
gs = model.group_size
if gs > 1:
    bg, _ = best_group_per_component(codes, model.n_groups, gs, masks, dgp.components)
    unit_atoms = lambda i, k: np.arange(bg[i] * gs, (bg[i] + 1) * gs)
else:
    absc = np.abs(codes)
    unit_atoms = lambda i, k: np.argsort(
        absc[masks[:, i]].mean(0) - absc[~masks[:, i]].mean(0))[-k:]

# One representative per interesting manifold type (curved first, then a Concept)
WANT = ['Sphere', 'Torus', 'SwissRoll', 'Mobius', 'Circle', 'Concept']
chosen = []
for want in WANT:
    for comp in dgp.components:
        if comp.mtype == want and masks[:, comp.idx].sum() > 200:
            chosen.append(comp); break

fig = plt.figure(figsize=(6.0, len(chosen) * 3.0))
for r, comp in enumerate(chosen):
    i = comp.idx
    act = np.flatnonzero(masks[:, i])
    m_true = m_te[act, i].astype(np.float64); mu = m_true.mean(0)
    _, _, Vt = np.linalg.svd(m_true - mu, full_matrices=False); V = Vt[:3]   # true basis
    th  = test['thetas'][i]
    col = th[:, 0] if th is not None and th.size else np.zeros(len(act))
    atoms = unit_atoms(i, comp.ambient_dim)
    rec = codes[act][:, atoms] @ dictionary[atoms]
    for c, (name, pts) in enumerate([('true mᵢ', m_true), ('recovered', rec)]):
        ax = fig.add_subplot(len(chosen), 2, r * 2 + c + 1, projection='3d')
        style_panel(ax, (pts - mu) @ V.T, col, CMAP)     # both in the TRUE basis
        if r == 0: ax.set_title(name, fontsize=12)
        if c == 0:
            ax.text2D(-0.12, 0.5, f'{comp.mtype}\n(dim {comp.ambient_dim})',
                      transform=ax.transAxes, fontsize=10, rotation=90, va='center', ha='center')
fig.suptitle('Per-concept recovery — TRUE 3D-PCA basis, colour = intrinsic θ₀', fontsize=13, y=1.0)
fig.tight_layout(); plt.show()